# 2. Exploratory Data Analysis

El análisis exploratorio se realiza **únicamente sobre `train.csv`**. Su propósito es comprender los datos y justificar decisiones posteriores, no modificar todavía las variables.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd "/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification"

/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"
for directory in (DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
target = contract["target"]
X_train = train_df.drop(columns=target)
y_train = train_df[target]
print(train_df.shape)
train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification/data/train.csv'

## Distribución de la variable objetivo

In [ ]:
counts = y_train.value_counts().sort_index()
ax = counts.plot.bar(color=["#c0392b", "#2980b9"])
ax.set_xticklabels(contract["target_names"], rotation=0)
ax.set_ylabel("Número de casos")
ax.set_title("Distribución de clases en entrenamiento")
plt.show()

## Distribuciones y escala de las variables

In [ ]:
X_train.describe().T[["mean", "std", "min", "50%", "max"]].sort_values("std", ascending=False).head(10)

In [ ]:
selected = ["mean radius", "mean texture", "mean perimeter", "mean area"]
train_df[selected].hist(figsize=(10, 7), bins=25, edgecolor="white")
plt.suptitle("Distribuciones de variables seleccionadas")
plt.tight_layout()
plt.show()

## Relación con la clase

In [ ]:
correlations = train_df.corr(numeric_only=True)[target].drop(target).abs().sort_values(ascending=False)
correlations.head(10).sort_values().plot.barh(color="#2c3e50")
plt.xlabel("|correlación con target|")
plt.title("Variables más relacionadas linealmente con la clase")
plt.show()

In [ ]:
top_features = correlations.head(8).index
plt.figure(figsize=(8, 6))
sns.heatmap(train_df[top_features].corr(), cmap="coolwarm", center=0)
plt.title("Correlaciones entre variables destacadas")
plt.tight_layout()
plt.show()

## Decisiones derivadas del EDA

- Las variables tienen escalas muy diferentes: regresión logística y KNN necesitan estandarización.
- Hay variables fuertemente correlacionadas. Esto no impide entrenar los modelos propuestos, pero debe considerarse al interpretar coeficientes e importancias.
- La clase presenta desbalance moderado; se utilizarán particiones estratificadas.
- No se observan valores faltantes, pero el pipeline incluirá imputación para soportar datos futuros incompletos.